In [0]:
# Databricks notebook source

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from io import StringIO
from datetime import datetime

# ============================================================================
# CONFIG
# ============================================================================

START_DATE = datetime(2020, 1, 1)
END_DATE = datetime.now()

TARGET_FUELS = ["WIND", "SOLAR"]

# ============================================================================
# HELPERS
# ============================================================================

def fetch_ieso_forecast_daily(start_date, end_date, target_hour=9):
    """
    Télécharge les prévisions VGForecastSummary jour par jour.
    Chaque fichier quotidien contient les prévisions pour les 48 prochaines heures.
    
    Args:
        start_date: Date de début
        end_date: Date de fin
        target_hour: Heure cible pour télécharger (défaut: 9h)
    
    Returns:
        DataFrame avec colonnes: ForecastTimestamp, TargetDatetime, FuelType, ForecastMW
    """
    import xml.etree.ElementTree as ET
    
    date_range = pd.date_range(start_date, end_date, freq='D')
    all_forecasts = []
    
    print(f"\n📥 Downloading VGForecastSummary (daily files, target hour: {target_hour}h)")
    
    for date in date_range:
        date_str = date.strftime('%Y%m%d')
        
        # Try version at target hour first, then without version (latest of the day)
        urls_to_try = [
            f"https://reports-public.ieso.ca/public/VGForecastSummary/PUB_VGForecastSummary_{date_str}_v{target_hour}.xml",
            f"https://reports-public.ieso.ca/public/VGForecastSummary/PUB_VGForecastSummary_{date_str}.xml"
        ]
        
        success = False
        for url in urls_to_try:
            try:
                r = requests.get(url, timeout=60)
                r.raise_for_status()
                
                # Parse XML
                root = ET.fromstring(r.content)
                ns = {'ieso': 'http://www.ieso.ca/schema'}
                
                doc_body = root.find('ieso:DocBody', ns)
                if doc_body is None:
                    continue
                
                forecast_timestamp_elem = doc_body.find('ieso:ForecastTimeStamp', ns)
                forecast_timestamp = forecast_timestamp_elem.text if forecast_timestamp_elem is not None else None
                
                rows = []
                
                # Parse structure: OrganizationData > FuelData > ResourceData > EnergyForecast > ForecastInterval
                # FILTRE: Garder uniquement MARKET PARTICIPANT + OntarioTotal
                for org_data in doc_body.findall('ieso:OrganizationData', ns):
                    # Extraire le type d'organisation
                    org_type_elem = org_data.find('ieso:OrganizationType', ns)
                    org_type = org_type_elem.text if org_type_elem is not None else None
                    
                    # FILTRE 1: Garder uniquement MARKET PARTICIPANT
                    if org_type != 'MARKET PARTICIPANT':
                        continue
                    
                    for fuel_data in org_data.findall('ieso:FuelData', ns):
                        fuel_type_elem = fuel_data.find('ieso:FuelType', ns)
                        fuel_type = fuel_type_elem.text if fuel_type_elem is not None else None
                        
                        for resource_data in fuel_data.findall('ieso:ResourceData', ns):
                            # Extraire la zone
                            zone_elem = resource_data.find('ieso:ZoneName', ns)
                            zone = zone_elem.text if zone_elem is not None else None
                            
                            # FILTRE 2: Garder uniquement OntarioTotal
                            if zone != 'OntarioTotal':
                                continue
                            
                            for energy_forecast in resource_data.findall('ieso:EnergyForecast', ns):
                                forecast_date_elem = energy_forecast.find('ieso:ForecastDate', ns)
                                forecast_date = forecast_date_elem.text if forecast_date_elem is not None else None
                                
                                for interval in energy_forecast.findall('ieso:ForecastInterval', ns):
                                    hour_elem = interval.find('ieso:ForecastHour', ns)
                                    mw_elem = interval.find('ieso:MWOutput', ns)
                                    
                                    if hour_elem is not None and mw_elem is not None and forecast_date:
                                        rows.append({
                                            'ForecastTimestamp': forecast_timestamp,
                                            'FuelType': fuel_type,
                                            'ForecastDate': forecast_date,
                                            'ForecastHour': int(hour_elem.text),
                                            'MWOutput': float(mw_elem.text)
                                        })
                
                if rows:
                    all_forecasts.extend(rows)
                    print(f"   {date_str}: {len(rows)} rows")
                    success = True
                    break
                    
            except Exception as e:
                continue
        
        if not success:
            print(f"   ⚠️ {date_str}: No data available")
    
    if not all_forecasts:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_forecasts)
    
    # Aggregate by FuelType + datetime (sum across all zones)
    df['TargetDatetime'] = pd.to_datetime(df['ForecastDate']) + pd.to_timedelta(df['ForecastHour'] - 1, unit='h')
    df['ForecastTimestamp'] = pd.to_datetime(df['ForecastTimestamp'])
    
    df = df.groupby(['ForecastTimestamp', 'TargetDatetime', 'FuelType'], as_index=False)['MWOutput'].sum()
    df.rename(columns={'MWOutput': 'ForecastMW'}, inplace=True)
    
    print(f"\n✅ Total forecast rows: {len(df):,}")
    
    return df


def fetch_ieso_csv(dataset, start_date, end_date):

    years = range(start_date.year, end_date.year + 1)

    dfs = []

    print(f"\n📥 Downloading {dataset}")

    for year in years:

        # Try CSV first
        url_csv = f"https://reports-public.ieso.ca/public/{dataset}/PUB_{dataset}_{year}.csv"
        # Fallback to XML if CSV fails
        url_xml = f"https://reports-public.ieso.ca/public/{dataset}/PUB_{dataset}_{year}.xml"

        df = None

        # Try CSV
        try:

            print(f"   {year} (CSV)")

            r = requests.get(
                url_csv,
                timeout=120
            )

            r.raise_for_status()

            df = pd.read_csv(
                StringIO(r.text),
                skiprows=3
            )

            df["SourceYear"] = year
            dfs.append(df)

        except Exception as e_csv:

            # Try XML as fallback
            try:

                print(f"   {year} (XML fallback)")

                r = requests.get(
                    url_xml,
                    timeout=120
                )

                r.raise_for_status()

                import xml.etree.ElementTree as ET
                root = ET.fromstring(r.content)

                # Parse IESO XML structure
                # Structure: Document -> DocBody -> DailyData -> HourlyData -> FuelTotal
                ns = {'ieso': 'http://www.ieso.ca/schema'}
                
                rows = []
                doc_body = root.find('ieso:DocBody', ns)
                
                if doc_body is not None:
                    for daily_data in doc_body.findall('ieso:DailyData', ns):
                        day_elem = daily_data.find('ieso:Day', ns)
                        day = day_elem.text if day_elem is not None else None
                        
                        for hourly_data in daily_data.findall('ieso:HourlyData', ns):
                            hour_elem = hourly_data.find('ieso:Hour', ns)
                            hour = hour_elem.text if hour_elem is not None else None
                            
                            for fuel_total in hourly_data.findall('ieso:FuelTotal', ns):
                                fuel_elem = fuel_total.find('ieso:Fuel', ns)
                                fuel = fuel_elem.text if fuel_elem is not None else None
                                
                                # EnergyValue contains OutputQuality and Output children
                                energy_value = fuel_total.find('ieso:EnergyValue', ns)
                                output = None
                                if energy_value is not None:
                                    output_elem = energy_value.find('ieso:Output', ns)
                                    output = output_elem.text if output_elem is not None else None
                                
                                rows.append({
                                    'Date': day,
                                    'Hour': hour,
                                    'Fuel Type': fuel,
                                    'Output (MW)': output
                                })

                if rows:
                    df = pd.DataFrame(rows)
                    df["SourceYear"] = year
                    dfs.append(df)
                else:
                    print(f"⚠️ {year} : No data in XML")

            except Exception as e_xml:

                print(f"⚠️ {year} : CSV failed ({e_csv}), XML failed ({e_xml})")

    if len(dfs) == 0:
        return pd.DataFrame()

    df = pd.concat(
        dfs,
        ignore_index=True
    )

    print(f"✅ {len(df):,} rows")

    return df


def create_ieso_datetime(date_col, hour_col):

    dt = (
        pd.to_datetime(date_col)
        + pd.to_timedelta(hour_col.astype(int) - 1, unit="h")
    )

    dt = (
        dt.dt.tz_localize(
            "America/Toronto",
            ambiguous=False,
            nonexistent="shift_forward"
        )
        .dt.tz_convert("UTC")
        .dt.tz_localize(None)
    )

    return dt


def rmse(actual, forecast):

    return np.sqrt(
        np.mean(
            (actual - forecast) ** 2
        )
    )

# ============================================================================
# LOAD ACTUAL GENERATION
# ============================================================================

gen = fetch_ieso_csv(
    "GenOutputbyFuelHourly",
    START_DATE,
    END_DATE
)

if gen.empty:
    raise Exception("No GenOutputbyFuelHourly data found")

gen.columns = (
    gen.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\nGEN COLUMNS")
print(gen.columns.tolist())

fuel_col = [c for c in gen.columns if "Fuel" in c][0]

output_candidates = [
    c for c in gen.columns
    if (
        "Output" in c
        or "EnergyMW" in c
        or c.lower().endswith("mw")
    )
]

output_col = output_candidates[-1]

gen["Date"] = pd.to_datetime(gen["Date"])
gen["Hour"] = pd.to_numeric(gen["Hour"])

gen["datetime"] = create_ieso_datetime(
    gen["Date"],
    gen["Hour"]
)

actual = (
    gen[
        [
            "datetime",
            fuel_col,
            output_col
        ]
    ]
    .rename(
        columns={
            fuel_col: "FuelType",
            output_col: "ActualMW"
        }
    )
)

actual["FuelType"] = (
    actual["FuelType"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Convert ActualMW to numeric (XML parsing returns strings)
actual["ActualMW"] = pd.to_numeric(
    actual["ActualMW"],
    errors="coerce"
)

actual = (
    actual[
        actual["FuelType"].isin(TARGET_FUELS)
    ]
    .groupby(
        ["datetime", "FuelType"],
        as_index=False
    )
    ["ActualMW"]
    .sum()
)

print(f"\n✅ Actual rows: {len(actual):,}")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✅ GENERATION DATA SUCCESSFULLY LOADED")
print("="*80)
print(f"\nTotal rows: {len(actual):,}")
print(f"Date range: {actual['datetime'].min()} to {actual['datetime'].max()}")
print(f"\nBreakdown by fuel type:")

summary = actual.groupby("FuelType").agg({
    "ActualMW": ["count", "mean", "min", "max", "sum"]
}).round(2)

display(summary)

print("\n" + "="*80)
print("⚠️  NOTE: VGForecastSummary not available from IESO")
print("="*80)
print("The forecast comparison analysis has been skipped.")
print("Only generation data (GenOutputbyFuelHourly) has been imported.\n")


In [0]:
# ============================================================================
# LOAD FORECAST DATA
# ============================================================================

# Les fichiers de prévisions IESO sont disponibles à partir du 1er août 2026
# Télécharge les prévisions quotidiennes (un fichier par jour vers 9h)
forecast_start = max(
    datetime(2026, 8, 1),  # Date de début des fichiers IESO
    END_DATE - pd.Timedelta(days=90)  # Maximum 90 jours en arrière
)

days_count = (END_DATE - forecast_start).days + 1
print(f"\n📅 Forecast period: {forecast_start.date()} to {END_DATE.date()} ({days_count} days)")

vg = fetch_ieso_forecast_daily(
    forecast_start,
    END_DATE,
    target_hour=9
)

if vg.empty:
    print("\n" + "="*80)
    print("⚠️  VGForecastSummary data not available")
    print("="*80)
    print("Forecast data could not be loaded.")
    print("If you have forecast files, you can load them here.\n")
else:
    # Les données sont déjà formatées par fetch_ieso_forecast_daily
    # Colonnes: ForecastTimestamp, TargetDatetime, FuelType, ForecastMW
    
    # Normaliser FuelType pour correspondre aux données actuelles
    vg['FuelType'] = vg['FuelType'].str.upper().str.strip()
    
    # Filtrer uniquement WIND et SOLAR
    vg = vg[vg['FuelType'].isin(TARGET_FUELS)]
    
    # Calculer le lead time (délai entre prévision et livraison)
    vg["LeadHours"] = (
        vg["TargetDatetime"]
        - vg["ForecastTimestamp"]
    ).dt.total_seconds() / 3600
    
    print(f"\n📈 Forecast statistics:")
    print(f"   Date range: {vg['TargetDatetime'].min()} to {vg['TargetDatetime'].max()}")
    print(f"   Lead time range: {vg['LeadHours'].min():.1f}h to {vg['LeadHours'].max():.1f}h")
    print(f"   Rows: {len(vg):,}")
    
    # ========================================================================
    # GARDER LES PRÉVISIONS JOUR-AHEAD (FAITES À 9H LA VEILLE)
    # ========================================================================
    # On garde les prévisions émises à 9h le jour X pour les heures du jour X+1
    # Lead time typique : ~24h à ~47h
    
    # Extraire la date de la prévision (sans l'heure)
    vg['ForecastDate'] = vg['ForecastTimestamp'].dt.date
    vg['TargetDate'] = vg['TargetDatetime'].dt.date
    
    # Garder uniquement les prévisions où TargetDate = ForecastDate + 1 jour
    vg['DayDiff'] = (vg['TargetDatetime'] - vg['ForecastTimestamp']).dt.days
    
    # Filtrer : prévision faite la veille (DayDiff = 1) ou début du surlendemain si heure < 9h
    # Pour être précis : garder les prévisions avec lead time entre 23h et 48h
    forecast = vg[
        (vg['LeadHours'] >= 23) & (vg['LeadHours'] <= 48)
    ].copy()
    
    print(f"\n✅ Forecast rows (day-ahead, 23-48h lead): {len(forecast):,}")
    print(f"   Lead time range: {forecast['LeadHours'].min():.1f}h to {forecast['LeadHours'].max():.1f}h")
    
    # ========================================================================
    # JOIN
    # ========================================================================
    
    df = forecast.merge(
        actual,
        left_on=[
            "TargetDatetime",
            "FuelType"
        ],
        right_on=[
            "datetime",
            "FuelType"
        ],
        how="inner"
    )
    
    print(f"\n✅ Matched rows: {len(df):,}")
    
    df["ErrorMW"] = (
        df["ForecastMW"]
        - df["ActualMW"]
    )
    
    df["AbsErrorMW"] = (
        df["ErrorMW"].abs()
    )
    
    df["APE"] = (
        df["AbsErrorMW"]
        /
        np.where(
            df["ActualMW"] == 0,
            np.nan,
            df["ActualMW"]
        )
    ) * 100
    
    # ========================================================================
    # KPI
    # ========================================================================
    
    kpis = (
        df.groupby("FuelType")
        .apply(
            lambda x: pd.Series({
    
                "MAE_MW":
                    round(
                        x["AbsErrorMW"].mean(),
                        2
                    ),
    
                "RMSE_MW":
                    round(
                        rmse(
                            x["ActualMW"],
                            x["ForecastMW"]
                        ),
                        2
                    ),
    
                "MAPE_%":
                    round(
                        x["APE"].mean(),
                        2
                    ),
    
                "BIAS_MW":
                    round(
                        x["ErrorMW"].mean(),
                        2
                    )
            })
        )
    )
    
    print("\n")
    print("=" * 80)
    print("FORECAST QUALITY")
    print("=" * 80)
    display(kpis)
    
    # ========================================================================
    # TIME SERIES
    # ========================================================================
    
    for fuel in TARGET_FUELS:
    
        sample = (
            df[
                df["FuelType"] == fuel
            ]
            .sort_values(
                "TargetDatetime"
            )
            .tail(24 * 30)
        )
    
        plt.figure(
            figsize=(18, 6)
        )
    
        plt.plot(
            sample["TargetDatetime"],
            sample["ActualMW"],
            label="Actual"
        )
    
        plt.plot(
            sample["TargetDatetime"],
            sample["ForecastMW"],
            label="Forecast"
        )
    
        plt.title(
            f"{fuel} Forecast vs Actual (Last 30 Days)"
        )
    
        plt.legend()
        plt.show()
    
    # ========================================================================
    # ERROR DISTRIBUTION
    # ========================================================================
    
    plt.figure(
        figsize=(12, 6)
    )
    
    sns.histplot(
        df,
        x="ErrorMW",
        hue="FuelType",
        bins=80,
        kde=True
    )
    
    plt.title("Forecast Error Distribution")
    plt.show()
    
    # ========================================================================
    # HEATMAP
    # ========================================================================
    
    df["Month"] = (
        df["TargetDatetime"]
        .dt.month
    )
    
    df["Hour"] = (
        df["TargetDatetime"]
        .dt.hour
    )
    
    for fuel in TARGET_FUELS:
    
        tmp = df[
            df["FuelType"] == fuel
        ]
    
        pivot = tmp.pivot_table(
            index="Hour",
            columns="Month",
            values="AbsErrorMW",
            aggfunc="mean"
        )
    
        plt.figure(
            figsize=(12, 8)
        )
    
        sns.heatmap(
            pivot,
            cmap="Reds"
        )
    
        plt.title(
            f"{fuel} Mean Absolute Error"
        )
    
        plt.show()
    
    # ========================================================================
    # TOP 5% ERRORS
    # ========================================================================
    
    threshold = (
        df["AbsErrorMW"]
        .quantile(0.95)
    )
    
    extreme_events = (
        df[
            df["AbsErrorMW"] >= threshold
        ]
        .sort_values(
            "AbsErrorMW",
            ascending=False
        )
    )
    
    print("\nTOP FORECAST ERRORS")
    
    display(
        extreme_events[
            [
                "TargetDatetime",
                "FuelType",
                "ForecastMW",
                "ActualMW",
                "ErrorMW",
                "AbsErrorMW"
            ]
        ]
    )